# Community simulation using SteadierCom

🎯 Learning objectives:

- Running SteadierCom and specifying options
- Loading simulation results with Pandas 
- Analysing cross-feeding interactions
- Drawing networks with pyvis

We will use as case-study the prediction of cross-feeding interactions in lignocellulose degrading communities as published in [Schäfer et al, 2026](https://www.nature.com/articles/s41522-026-01072-x). 

In particular we will try to replicate the simulation results (compost isolates enriched in xylan medium) presented in [Figure 7](https://www.nature.com/articles/s41522-026-01072-x/figures/7):

![Fig 7](files2/fig7.png)

### Step 0. Testing your setup

[SteadierCom](https://github.com/cdanielmachado/steadiercom) is a command-line tool. Remember, we can run terminal commands directly from jupyter notebooks using `!` : 

In [ ]:
! steadiercom -h

### Step 1. Running a simulation



**SteadierCom** will require several inputs:

- Mandatory:
  - Genome-scale metabolic models for each member
- Optional:
  - Relative abundances 
  - Community growth rate 
  - Growth media
  - "Unlimited" compounds (like water, minerals, trace metals) to exclude from analysis
  - Sample size (for sampling the solution space)

We are now ready to run (this can take a few minutes):

In [ ]:
!steadiercom files2/models/*.xml \
   -c files2/abundances.tsv \
   --growth 0.01 \
   -m xylan --mediadb files2//media_db.tsv \
   --unlimited files2/unlimited.txt \
   --sample 100 

(When the simulation is finished you should see a new file called [**output.tsv**](files2/output.tsv).

### Step 2. Loading our results

The output of **SteadierCom** is a table in TSV format that you can inspect with a text editor or open in a spreadsheet software.

In this tutorial, we will use the powerful [**pandas**](https://pandas.pydata.org/) library.

> ⚠️ If you never used **pandas** before, please take a few minutes to get familiar with the [documentation](https://pandas.pydata.org/docs/user_guide/index.html).

Let's start by loading and inspecting the file:

In [ ]:
import pandas as pd

df = pd.read_csv('files2/output.tsv', sep='\t')
df.sample(10) # show 10 randomly selected entries

This table as the following columns:

- **donor**: community member (or environment)
- **receiver**: community member (or environment)
- **compound**: the metabolite being exchanged
- **mass_rate**: exchange flux in grams(compound)/gDW/h
- **rate**: exchange flux in mmol/gDW/h
- **frequency**: fraction of samples where interaction occurs
- **community**: relevant if we simulate multiple communities
- **medium**: relevant if we simulate growth in multiple media

### Step 3: filtering

We can use the `.query()` and `.sort()` methods to find and sort entries.

Let's check who is consuming compounds directly from the growth medium, and sort them by consumption rate.

In [ ]:
df.query('donor == "environment"').sort_values('mass_rate', ascending=False)

You can see that many interactions are predicted to occur at a very low rate or with low frequency.

Let's filter for interactions with:

  - exchange rate at least 0.1 mg/gDW/h
  - frequency of at least 10% 

In [ ]:
df = df.query('mass_rate > 1e-4 and frequency > 0.1')

Let's check how many entries are left...

> 👉 Note that since we used random sampling to explore the solution space, your results will be slightly different every time.

In [ ]:
print(f'Number of rows: {df.shape[0]}')

### Step 4 - Querying

In the figure you can see that **Lacrimospora** is the most abundant member and the main donor. 

Let's check what it might be donating to the other members:

In [ ]:
df.query('donor == "Lacrimospora" and receiver != "environment"').sort_values(['receiver'])

🤔 What else can we search for?

Take some time to explore by yourself...

In [ ]:
# type your code here...

### Drawing a network with pyvis

We can use the [**pyvis**](https://pyvis.readthedocs.io/en/latest/) library to display an interaction network.

For convenience, here is a function that will convert our table into a network:

In [ ]:
from pyvis.network import Network

def build_network(df):

    df = df.query('donor != "environment" and receiver != "environment"')
    
    net = Network(directed=True, height='500px', width='800px', 
                  notebook=True,  cdn_resources='in_line')
    
    species = set(df['donor']) | set(df['receiver'])
    net.add_nodes(species)
    
    for cpd in set(df['compound']):
        net.add_node(cpd[2:-2], shape='box')
    
    for _, row in df.iterrows():
        width = row['mass_rate'] * row['frequency']
        net.add_edge(row['donor'], row['compound'][2:-2], value=width)
        net.add_edge(row['compound'][2:-2], row['receiver'], value=width)
    
    return net.show('tmp.html')

Let's see how that works:

> ⚠️ Note: Sometimes Jupyter doesn't display the network correctly. If that happens you can open the [tmp.html](tmp.html) file directly.

In [ ]:
build_network(df)

You can click and drag around the nodes to explore the interactions.

Most likely the network looks like a "hairball". 

Try to explore by changing the cutoff values for the rates and frequencies and re-draw the network. 

In [ ]:
# Type your code here... 

---------

Congratulations, you reached the end of the tutorial. 🥳

If you finished early, feel free to ask questions or try to help another colleague... 🙂